# TP5. Synthèse : chiffrer, déchiffrer, casser

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp5-synthese/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : les deux conversions

Écrivez `lettre_vers_nombre(c)` et `nombre_vers_lettre(n)`, réciproques l'une de
l'autre sur les majuscules.

**Résultat attendu :** `lettre_vers_nombre("C")` vaut `2`, et
`nombre_vers_lettre(2)` vaut `"C"`.


In [ ]:
def lettre_vers_nombre(c):
    return ord(c) - ord("A")


def nombre_vers_lettre(n):
    return chr(ord("A") + n % 26)


print(lettre_vers_nombre("C"))
print(nombre_vers_lettre(2))

## Exercice 1.2 : nettoyer un texte

Écrivez `nettoyer(texte)` qui met tout en majuscules et ne garde que les lettres de A
à Z, en supprimant espaces, ponctuation et accents.

**Résultat attendu :** `nettoyer("Été, à Paris !")` vaut `"ETEAPARIS"`.


In [ ]:
import unicodedata


def nettoyer(texte):
    texte = unicodedata.normalize("NFD", texte.upper())
    return "".join(c for c in texte if "A" <= c <= "Z")


print(nettoyer("Été, à Paris !"))

## Exercice 2.1 : la fonction de chiffrement

Écrivez `chiffrer(texte, a, b)` qui applique $E(x) = (ax + b) \bmod 26$ à chaque
lettre d'un texte déjà nettoyé.

**Résultat attendu :** `chiffrer("ATTAQUEALAUBE", 5, 8)` vaut `"IZZIKECILIENC"`.


In [ ]:
def chiffrer(texte, a, b):
    resultat = ""
    for lettre in texte:
        nombre = lettre_vers_nombre(lettre)
        resultat += nombre_vers_lettre(a * nombre + b)
    return resultat


print(chiffrer("ATTAQUEALAUBE", 5, 8))

## Exercice 3.1 : les clés utilisables

Écrivez le code qui donne la liste des valeurs de `a` entre 1 et 25 pour lesquelles le
chiffrement est déchiffrable. Combien y en a-t-il ?

**Résultat attendu :** douze valeurs, toutes impaires et différentes de 13.


In [ ]:
import math


cles_a = [a for a in range(1, 26) if math.gcd(a, 26) == 1]
print(cles_a)
print("Nombre de valeurs :", len(cles_a))

## Exercice 3.2 : l'inverse modulaire

Écrivez `inverse_modulaire(a, n)` qui renvoie l'inverse de `a` modulo `n`, ou lève une
erreur s'il n'existe pas.

**Résultat attendu :** l'inverse de 5 modulo 26 est 21, car $5 \times 21 = 105 =
4 \times 26 + 1$.


In [ ]:
def inverse_modulaire(a, n):
    for inverse in range(n):
        if (a * inverse) % n == 1:
            return inverse
    raise ValueError("Cet inverse n'existe pas")


print(inverse_modulaire(5, 26))

## Exercice 4.1 : la fonction de déchiffrement

Écrivez `dechiffrer(texte, a, b)`, en refusant les clés invalides. Vérifiez que
déchiffrer ce qu'on a chiffré redonne le message d'origine.

**Résultat attendu :** `dechiffrer(chiffrer(m, 5, 8), 5, 8)` redonne `m` nettoyé.


In [ ]:
def dechiffrer(texte, a, b):
    if math.gcd(a, 26) != 1:
        raise ValueError("La clé a ne permet pas de déchiffrer")

    inverse = inverse_modulaire(a, 26)
    resultat = ""
    for lettre in texte:
        nombre = lettre_vers_nombre(lettre)
        resultat += nombre_vers_lettre(inverse * (nombre - b))
    return resultat


message = nettoyer("Attaque à l'aube !")
secret = chiffrer(message, 5, 8)
print(secret)
print(dechiffrer(secret, 5, 8))
assert dechiffrer(secret, 5, 8) == message

## Exercice 5.1 : essayer toutes les clés

Écrivez `toutes_les_clefs(secret)` qui renvoie la liste des 312 déchiffrements
possibles avec leur clé. Retrouvez le message clair à l'œil.

**Résultat attendu :** parmi les 312 propositions, une seule est du français lisible.


In [ ]:
def toutes_les_clefs(secret):
    propositions = []
    for a in range(1, 26):
        if math.gcd(a, 26) == 1:
            for b in range(26):
                propositions.append((a, b, dechiffrer(secret, a, b)))
    return propositions


texte_long = nettoyer("""Dans le village, les habitants se retrouvent autour de la grande place.
Le marche ouvre chaque matin, et chacun apporte des fruits, du pain et des nouvelles.
Une lettre arrive parfois de la ville voisine, puis le facteur repart sur son velo.
Les enfants jouent pres de la fontaine tandis que les anciens racontent leurs voyages.
Quand le soleil descend, les familles rentrent chez elles et le calme revient.
Ce petit recit parle du temps qui passe, des liens entre les gens et des choses simples
que l on garde en memoire.""")
secret_long = chiffrer(texte_long, 5, 8)
propositions = toutes_les_clefs(secret_long)

print("Nombre de propositions :", len(propositions))
print("Message clair :", next(p for p in propositions if p[2] == texte_long))

## Exercice 6.1 : casser automatiquement

Comptez les occurrences de chaque lettre du message chiffré, puis testez les clés qui
envoient E sur la lettre la plus fréquente. Retrouvez le message sans intervention
humaine.

**Résultat attendu :** le message clair, trouvé parmi une poignée de candidats au
lieu de 312.


In [ ]:
from collections import Counter


frequences = Counter(secret_long)
lettre_frequente = frequences.most_common(1)[0][0]
numero_frequent = lettre_vers_nombre(lettre_frequente)

print("Lettre chiffrée la plus fréquente :", lettre_frequente)
print("Clés possibles et débuts des messages :")
for a in cles_a:
    b = (numero_frequent - 4 * a) % 26
    candidat = dechiffrer(secret_long, a, b)
    print((a, b), candidat[:60])

## Exercice 7.1 : l'indice de coïncidence

Écrivez `indice_coincidence(texte)`. Calculez-le sur `texte_long`, sur `secret_long`,
et sur une suite de lettres tirées au hasard de même longueur.

**Résultat attendu :** la même valeur pour le clair et pour le chiffré affine, environ
0,087 sur ce texte court, contre environ 0,039 pour les lettres au hasard.


In [ ]:
def indice_coincidence(texte):
    n = len(texte)
    if n < 2:
        return 0

    frequences = Counter(texte)
    numerateur = sum(nombre * (nombre - 1) for nombre in frequences.values())
    return numerateur / (n * (n - 1))


import random

texte_aleatoire = "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ", k=len(texte_long)))
print("Texte clair :", indice_coincidence(texte_long))
print("Texte chiffré :", indice_coincidence(secret_long))
print("Lettres aléatoires :", indice_coincidence(texte_aleatoire))

## Exercice 8.1 : chiffrer par Vigenère

Écrivez `vigenere(texte, cle)`. Chiffrez `texte_long` avec la clé `"CLE"`, et calculez
l'indice de coïncidence du résultat.

**Résultat attendu :** un indice d'environ 0,055, intermédiaire entre le français et
le hasard. L'attaque par fréquences de l'étape 6 ne s'applique plus telle quelle.


In [ ]:
def vigenere(texte, cle):
    cle = nettoyer(cle)
    if not cle:
        raise ValueError("La clé ne peut pas être vide")

    resultat = ""
    for i, lettre in enumerate(texte):
        decalage = lettre_vers_nombre(cle[i % len(cle)])
        resultat += nombre_vers_lettre(lettre_vers_nombre(lettre) + decalage)
    return resultat


secret_vigenere = vigenere(texte_long, "CLE")
print(secret_vigenere)
print("Indice de coïncidence :", indice_coincidence(secret_vigenere))

## Exercice 8.2 : retrouver la longueur de la clé

Si la clé a $L$ lettres, alors les lettres du chiffré de rangs $0, L, 2L, \dots$
ont toutes subi le même décalage : c'est un chiffre de César, et son indice de
coïncidence est celui du français. Pour chaque longueur candidate `L` de 1 à 7,
découpez le chiffré en `L` colonnes, calculez l'indice moyen des colonnes, et repérez
les longueurs où il remonte.

**Résultat attendu :** l'indice moyen bondit à environ 0,09 pour `L = 3` et
`L = 6`, et reste vers 0,055 pour les autres longueurs. La clé a donc trois lettres,
ou un multiple de trois.


In [ ]:
for longueur in range(1, 8):
    indices_colonnes = [
        indice_coincidence(secret_vigenere[i::longueur])
        for i in range(longueur)
    ]
    moyenne = sum(indices_colonnes) / longueur
    print(f"Longueur {longueur} : indice moyen {moyenne:.3f}")